# Apex Predictor — Deep Learning: LSTM sulla sequenza di gare

Secondo esperimento. Invece di feature riassuntive (media/mediana), diamo alla rete la sequenza grezza delle ultime 10 gare di ciascun pilota (griglia, posizione, punti), lasciando che una LSTM impari da sola i pattern temporali. Architettura ibrida: ramo LSTM (sequenza) + ramo MLP (feature statiche della gara corrente).

Confronto di riferimento: XGBoost F1 0.717, MLP+embedding F1 0.657.

In [21]:
import sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

from src.data_loading import load_raw_data, build_working_dataset
from src.features import build_all_features
from src.train import FEATURE_COL, temporal_split
from src.dl_common import build_driver_sequences, find_best_threshold_np

torch.manual_seed(42)

data = load_raw_data()
df = build_working_dataset(data["races"], data["results"], min_year=2004)
df = build_all_features(
    df, data["circuits"], data["drivers"], data["constructors"],
    data["driver_standings"], data["constructor_standings"], data["qualifying"]
)
df = df.reset_index(drop=True)  # indici puliti 0..N, necessari per allineare sequences

SEQ_LEN = 10
sequences, seq_lengths = build_driver_sequences(df, seq_len=SEQ_LEN)
print(f"Sequenze costruite: {sequences.shape}, lunghezze storiche: min={seq_lengths.min()}, max={seq_lengths.max()}")

STATIC_FEATURES = [f for f in FEATURE_COL if f not in ["driver_recent_points_avg", "driver_recent_position_avg"]]
print(f"Feature statiche usate: {len(STATIC_FEATURES)} (escluse le 2 di forma, ora sostituite dalla sequenza)")

Sequenze costruite: (9278, 10, 3), lunghezze storiche: min=1, max=10
Feature statiche usate: 12 (escluse le 2 di forma, ora sostituite dalla sequenza)


In [22]:
train_df, test_df = temporal_split(df)
train_idx, test_idx = train_df.index.values, test_df.index.values

# Normalizzazione feature statiche
static_mean = df.loc[train_idx, STATIC_FEATURES].mean()
static_std = df.loc[train_idx, STATIC_FEATURES].std().replace(0, 1)

X_train_static = ((df.loc[train_idx, STATIC_FEATURES] - static_mean) / static_std).values
X_test_static = ((df.loc[test_idx, STATIC_FEATURES] - static_mean) / static_std).values

# Normalizzazione feature di sequenza: calcoliamo media/std sui valori reali del train (non sul padding), poi applichiamo a tutto
seq_features_flat_train = sequences[train_idx][seq_lengths[train_idx] > 0]
seq_mean = seq_features_flat_train.reshape(-1, 3).mean(axis=0)
seq_std = seq_features_flat_train.reshape(-1, 3).std(axis=0)
seq_std[seq_std == 0] = 1

sequences_norm = (sequences - seq_mean) / seq_std

X_train_seq = sequences_norm[train_idx]
X_test_seq = sequences_norm[test_idx]
train_lengths = seq_lengths[train_idx]
test_lengths = seq_lengths[test_idx]

y_train = train_df["podium"].astype(float).values
y_test = test_df["podium"].astype(float).values

print(f"Train: {len(y_train)} righe, Test: {len(y_test)} righe")

Train: 8519 righe, Test: 759 righe


In [23]:
class HybridDataset(Dataset):
    def __init__(self, X_seq, lengths, X_static, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.lengths[idx], self.X_static[idx], self.y[idx]


class LSTMHybrid(nn.Module):
    """
    Ramo LSTM (elabora la sequenza storica del pilota) + ramo MLP (elabora le feature statiche della gara corrente), uniti prima del livello di decisione finale.
    """
    def __init__(self, seq_input_dim, static_dim, lstm_hidden=16, static_hidden=32):
        super().__init__()
        # batch_first=True: i tensori hanno forma (batch, seq_len, features) invece di (seq_len, batch, features), più intuitivo
        self.lstm = nn.LSTM(seq_input_dim, lstm_hidden, batch_first=True)

        self.static_net = nn.Sequential(
            nn.Linear(static_dim, static_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden + static_hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x_seq, lengths, x_static):
        # pack_padded_sequence dice alla LSTM di ignorare i timestep di padding, enforce_sorted=False perché non abbiamo ordinato il batch per lunghezza decrescente
        packed = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        seq_repr = h_n[-1]  # stato nascosto finale = "riassunto" imparato della sequenza

        static_repr = self.static_net(x_static)

        combined = torch.cat([seq_repr, static_repr], dim=1)
        return self.head(combined).squeeze(-1)


train_dataset = HybridDataset(X_train_seq, train_lengths, X_train_static, y_train)
test_dataset = HybridDataset(X_test_seq, test_lengths, X_test_static, y_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

model = LSTMHybrid(seq_input_dim=3, static_dim=len(STATIC_FEATURES))
print(model)

LSTMHybrid(
  (lstm): LSTM(3, 16, batch_first=True)
  (static_net): Sequential(
    (0): Linear(in_features=12, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=48, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [24]:
class HybridDataset(Dataset):
    def __init__(self, X_seq, lengths, X_static, y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.lengths[idx], self.X_static[idx], self.y[idx]


class LSTMHybrid(nn.Module):
    """
    Ramo LSTM (elabora la sequenza storica del pilota) + ramo MLP (elabora le feature statiche della gara corrente), uniti prima del livello di decisione finale.
    """
    def __init__(self, seq_input_dim, static_dim, lstm_hidden=16, static_hidden=32):
        super().__init__()
        # batch_first=True: i tensori hanno forma (batch, seq_len, features) invece di (seq_len, batch, features), più intuitivo, coerente con come li abbiamo costruiti
        self.lstm = nn.LSTM(seq_input_dim, lstm_hidden, batch_first=True)

        self.static_net = nn.Sequential(
            nn.Linear(static_dim, static_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        self.head = nn.Sequential(
            nn.Linear(lstm_hidden + static_hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x_seq, lengths, x_static):
        # pack_padded_sequence dice alla LSTM di ignorare i timestep di padding
        # enforce_sorted=False perché non abbiamo ordinato il batch per lunghezza decrescente
        packed = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        seq_repr = h_n[-1]  # stato nascosto finale = "riassunto" imparato della sequenza

        static_repr = self.static_net(x_static)

        combined = torch.cat([seq_repr, static_repr], dim=1)
        return self.head(combined).squeeze(-1)


train_dataset = HybridDataset(X_train_seq, train_lengths, X_train_static, y_train)
test_dataset = HybridDataset(X_test_seq, test_lengths, X_test_static, y_test)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

model = LSTMHybrid(seq_input_dim=3, static_dim=len(STATIC_FEATURES))
print(model)

LSTMHybrid(
  (lstm): LSTM(3, 16, batch_first=True)
  (static_net): Sequential(
    (0): Linear(in_features=12, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=48, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [25]:
print("train_lengths min:", train_lengths.min())
print("test_lengths min:", test_lengths.min())

train_lengths min: 1
test_lengths min: 1


In [26]:
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor(n_neg / n_pos)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x_seq, lengths, x_static, y in loader:
            logits = model(x_seq, lengths, x_static)
            total_loss += criterion(logits, y).item()
    return total_loss / len(loader)


N_EPOCHS = 100
patience = 10
best_test_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    for x_seq, lengths, x_static, y in train_loader:
        optimizer.zero_grad()
        logits = model(x_seq, lengths, x_static)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    test_loss = evaluate_loss(model, test_loader, criterion)

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        epochs_without_improvement = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{N_EPOCHS} — train loss: {epoch_loss/len(train_loader):.4f}, test loss: {test_loss:.4f}")

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping all'epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)
print(f"Miglior test loss: {best_test_loss:.4f}")

Epoch 5/100 — train loss: 0.6194, test loss: 0.5559
Epoch 10/100 — train loss: 0.6036, test loss: 0.5424
Epoch 15/100 — train loss: 0.5867, test loss: 0.5284
Epoch 20/100 — train loss: 0.5730, test loss: 0.5326
Epoch 25/100 — train loss: 0.5627, test loss: 0.5309
Epoch 30/100 — train loss: 0.5561, test loss: 0.5219
Epoch 35/100 — train loss: 0.5502, test loss: 0.5311
Epoch 40/100 — train loss: 0.5483, test loss: 0.5307

Early stopping all'epoch 41
Miglior test loss: 0.5199


In [27]:
model.eval()
all_probas = []
with torch.no_grad():
    for x_seq, lengths, x_static, y in test_loader:
        logits = model(x_seq, lengths, x_static)
        probas = torch.sigmoid(logits)
        all_probas.append(probas.numpy())

y_proba = np.concatenate(all_probas)
best = find_best_threshold_np(y_test, y_proba)
print(f"LSTM ibrida — soglia {best['threshold']:.2f}: "
      f"precision {best['precision']:.3f}, recall {best['recall']:.3f}, F1 {best['f1']:.3f}")
print(f"\nConfronto — XGBoost: F1 0.717 | MLP+embedding: F1 0.657")

LSTM ibrida — soglia 0.80: precision 0.655, recall 0.703, F1 0.678

Confronto — XGBoost: F1 0.717 | MLP+embedding: F1 0.657
